# Proyecto: 120 years of Olympic history
Analisis exploratorio de datos usando Pandas

**Integrantes:**
- Julian Andres Velandia Cortes

## 1. Cargar cada conjunto de datos 

In [ ]:
import pandas as pd

df_athlete = pd.read_csv('../data/athlete_events.csv')
df_regions = pd.read_csv('../data/noc_regions.csv')

## 2. Descripcion de las variables

**Dataset athlete_events.csv:**
* **ID:** Identificador de cada atleta.
* **Name:** Nombre del atleta.
* **Sex:** Sexo del atleta (M o F).
* **Age:** Edad del atleta en anios.
* **Height:** Estatura en centimetros.
* **Weight:** Peso en kilogramos.
* **Team:** Nombre del equipo o pais.
* **NOC:** Codigo de 3 letras del Comite Olimpico Nacional.
* **Games:** Anio y temporada de los juegos.
* **Year:** Anio de la edicion.
* **Season:** Temporada (Summer o Winter).
* **City:** Ciudad sede.
* **Sport:** Deporte general.
* **Event:** Prueba especifica en la que compitio.
* **Medal:** Medalla ganada (Gold, Silver, Bronze o vacio/NaN).

**Dataset noc_regions.csv:**
* **NOC:** Codigo de 3 letras del Comite Olimpico Nacional.
* **region:** Nombre del pais o region geografica.
* **notes:** Notas adicionales sobre la region o territorio.

## 3. Estadisticas basicas

In [ ]:
# 3. generar las estadísticas basicas de cada dataframe y comparaciones
display(df_athlete.describe(include='all'))
display(df_regions.describe(include='all'))

**Comparacion entre los dataframes:**
El dataframe ""df_athlete"" tiene 271,116 filas y 15 columnas (combina variables numericas como edad, peso, estatura y ano con variables de texto), donde cada fila es la participacion de un atleta en una prueba y existen 226 codigos NOC distintos. El dataframe ""df_regions"" tiene 230 filas y 3 columnas (todas de texto), donde cada fila representa un codigo NOC unico con su respectiva region.

## 4. Creación de la variable IMC
Fórmula: Peso / (Estatura/100)^2

In [ ]:

df_athlete['IMC'] = df_athlete['Weight'] / ((df_athlete['Height'] / 100) ** 2)
df_athlete[['Weight', 'Height', 'IMC']].head()

**¿Qué opina de esta nueva variable? ¿Tiene el mismo sentido en todos los deportes?**

El IMC sirve para darnos una idea general de la relacion entre el peso y la estatura de una persona, pero en atletas olimpicos no es una medida muy confiable para evaluar su estado fisico porque la formula no diferencia entre grasa y masa muscular.

Por eso mismo, no tiene el mismo sentido en todos los deportes. Cada disciplina exige un tipo de cuerpo muy distinto: por ejemplo, un atleta de levantamiento de pesas o de lucha va a tener un IMC muy alto (que en una tabla normal saldria como "sobrepeso"), pero en realidad ese peso es puro musculo necesario para su deporte. En cambio, deportistas de gimnasia o maraton suelen tener un IMC mucho mas bajo porque necesitan ser ligeros. Compararlos a todos con la misma regla del IMC no refleja su verdadero rendimiento ni su salud.

## 5. Fusion (merge) de atletas y regiones

In [ ]:
#fucionar el dataframe de atletas con el de regiones usando NOC
df_merged = pd.merge(df_athlete, df_regions, on='NOC', how='left')

# verificar el numero de filas antes y despues del cruce
print("Filas antes del merge:", len(df_athlete))
print("Filas despues del merge:", len(df_merged))

# revisar cuantos registros quedan sin region asociada
sin_region = df_merged['region'].isna().sum() # esto de .isna() revisa que celdas estan vacias 
print("Filas sin region:", sin_region)
print("Codigos NOC sin region:", df_merged[df_merged['region'].isna()]['NOC'].unique()) #esto del .unique() es para obtener los valores distintos (sin repetir)

* **¿Por que cree que ocurre esto?** Quedan filas sin region porque en el dataset de atletas existen codigos NOC historicos o especiales (como "ROT" de refugiados o "TUV" de Tuvalu) que no tienen un pais asociado en la tabla pequena de regiones.
* **¿Que tipo de fusion (how) utilizo y por que?** Se utilizo "how='left'". Esto garantiza que se mantengan absolutamente todas las filas del dataframe principal de atletas sin perder ningun registro, trayendo la region cuando coincide el codigo NOC y dejando vacio ("NaN") donde no hay coincidencia.

## 6. Creacion de la variable binaria de medallas

In [ ]:
#preguntar si la columna 'Medal' NO esta vacia
#con el .astype(int) convertimos ese Verdadero/Falso en 1 y 0.
df_merged['tiene_medalla'] = df_merged['Medal'].notna().astype(int)

# mostrar  los primeros resultados para ver si quedo bien
df_merged[['Name', 'Sport', 'Medal', 'tiene_medalla']].head(10)

Con esta nueva columna binaria ("1" o "0"), podemos sumar facilmente cuantos podios ha conseguido un pais, un deporte o un atleta especifico utilizando operaciones basicas de Pandas, sin que nos estorben los valores vacios ("NaN").

## 7. Analisis de representatividad de los datos

In [ ]:

# Ver el primer y ultimo anio registrado (.min y .max sacan el menor y mayor valor)
print("anio minimo:", df_merged['Year'].min())
print("anio maximo:", df_merged['Year'].max())

# Contar cuantos paises y deportes distintos hay (.nunique cuenta valores unicos sin repetir)
print("Total de regiones distintas:", df_merged['region'].nunique())
print("Total de deportes distintos:", df_merged['Sport'].nunique())

# Contar cuantos registros hay por sexo (.value_counts cuenta cuantas veces sale cada opcion)
print(df_merged['Sex'].value_counts())

# Contar cuantos registros hay en juegos de veranio vs invierno
print(df_merged['Season'].value_counts())

# Mostrar los 5 paises con mas participaciones
print(df_merged['region'].value_counts().head(5))

**Analisis de representatividad:**
El conjunto de datos cubre 120 anos de historia (1896 a 2016) con 205 regiones y 66 deportes, pero no es completamente equilibrado en todos los aspectos:
* **Por sexo:** Hay mucha mas participacion de hombres que de mujeres, ya que en las primeras ediciones de los Juegos Olimpicos las mujeres casi no podian competir.
* **Por temporada:** Hay muchos mas datos de los juegos de verano ("Summer") que de invierno ("Winter"), porque los de verano empezaron antes y tienen mas deportes.
* **Por pais:** Potencias como Estados Unidos, Francia, Reino Unido e Italia concentran una gran parte de los registros, mientras que paises mas pequenos tienen muy pocas participaciones.

## 8. Analisis de datos faltantes

In [ ]:

faltantes = df_merged.isna().sum()

# Calcular el porcentaje de datos faltantes respecto al total de filas
# len(df_merged) nos da el total de filas
porcentaje = (faltantes / len(df_merged) * 100)

# Juntar ambos resultados en una tabla pequena para verlos facil
tabla_faltantes = pd.DataFrame({
    'Cantidad_faltantes': faltantes,
    'Porcentaje': porcentaje
})

# las columnas que tienen mas de 1 dato faltante
tabla_faltantes[tabla_faltantes['Cantidad_faltantes'] > 0]

**Analisis de datos faltantes (Punto 8):**
* **Medal (85.33%):** Es la que mas vacios tiene, pero no es un error. Significa simplemente que ese atleta no gano medalla en esa prueba.
* **notes (98.13%):** Casi toda esta vacia porque solo se llena para regiones especiales que necesitan una aclaracion historica.
* **Height, Weight, IMC y Age:** Faltan sobre todo en las primeras ediciones de los Juegos Olimpicos (finales de 1800 y principios de 1900), cuando no se registraban las medidas fisicas de todos los deportistas.
* **region (0.14%):** Son los pocos codigos NOC especiales que vimos en el punto 5.

**¿Que decision tomamos con estos faltantes?**
No conviene usar "dropna()" para borrar las filas vacias, porque borrariamos a todos los atletas que no ganaron medalla y gran parte de la historia antigua de los juegos. Lo mejor es dejar los valores vacios como estan: para "Medal" ya creamos la columna "tiene_medalla" (1 y 0), y para los calculos numericos de edad, peso, estatura e IMC, Pandas ignora automaticamente los "NaN" sin dañar los promedios.

## 9. Agrupaciones por pais, sexo y deporte

In [ ]:
#9.1
# .groupby('region') junta todas las filas que pertenecen al mismo pais
# ['tiene_medalla'].sum() suma los 1 y 0 de la columna para contar las medallas
medallas_pais = df_merged.groupby('region')['tiene_medalla'].sum().sort_values(ascending=False)

print("Top 10 paises con mas medallas:")
print(medallas_pais.head(10))

# 9.2 
# .mean() calcula el promedio de las columnas
promedios_sexo = df_merged.groupby('Sex')[['Age', 'Height', 'Weight', 'IMC']].mean()

print("\nPromedios fisicos por sexo:")
display(promedios_sexo)

# 9.3 Top 5 deportes con el IMC promedio mas alto
imc_deporte = df_merged.groupby('Sport')['IMC'].mean().sort_values(ascending=False)

print(imc_deporte.head(5))

**Analisis de las agrupaciones (Punto 9):**
* **Medallas por pais:** Estados Unidos lidera el medallero historico con mucha ventaja, seguido por Rusia, Alemania, Reino Unido e Italia. Esto coincide con lo visto en el punto 7, ya que son los paises con mayor cantidad de participaciones.
* **Promedios por sexo:** En promedio, los atletas hombres registran mayor estatura, peso e IMC que las mujeres, y tambien una edad promedio ligeramente superior.
* **IMC por deporte:** Los deportes con mayor IMC promedio son disciplinas de fuerza y combate (como tira y afloja, levantamiento de pesas y judo). Esto confirma lo que concluimos en el punto 4: un IMC alto en atletas olimpicos suele reflejar alta masa muscular y no sobrepeso.

## 10. Agrupaciones multiples y tabla de medallas por tipo

In [ ]:
# 10.1 
# pd.crosstab() crea una tabla cruzada contando cuantas veces se repite cada combinacion
medallero = pd.crosstab(df_merged['region'], df_merged['Medal'])

# Creamos una columna nueva 'Total' sumando las tres columnas de medallas
medallero['Total'] = medallero['Gold'] + medallero['Silver'] + medallero['Bronze']

# Ordenamos de mayor a menor segun las medallas de oro ('Gold')
medallero = medallero.sort_values(by='Gold', ascending=False)

display(medallero[['Gold', 'Silver', 'Bronze', 'Total']].head(10))

# 10.2 Agrupacion doble: participaciones y medallas por temporada y sexo
# Esto .agg() permite pedir dos calculos distintos al tiempo: contar filas ('count') y sumar medallas ('sum')
resumen_temporada_sexo = df_merged.groupby(['Season', 'Sex'])['tiene_medalla'].agg(['count', 'sum'])

# Renombramos las columnas para que se entiendan mejor
resumen_temporada_sexo.columns = ['Total_participaciones', 'Total_medallas']

display(resumen_temporada_sexo)

**Analisis de agrupaciones multiples (Punto 10):**
* **Medallero por tipo:** Estados Unidos domina claramente tanto en el total de medallas como en medallas de oro, seguido por Rusia y Alemania. Al separar por tipo de medalla se ve que los paises potencia mantienen una proporcion equilibrada entre oro, plata y bronce.
* **Temporada y sexo:** Tanto en los juegos de verano ("Summer") como en los de invierno ("Winter"), los hombres tienen mas participaciones y medallas totales que las mujeres debido a la historia temprana de los juegos. Sin embargo, al comparar la cantidad de medallas frente al total de participaciones, la proporcion de exito es muy similar entre ambos sexos.